# 05 — ETL Penitenciaría (Idescat)

Construye `contexto_penitenciaria` combinando los 4 ficheros históricos de Idescat (2010-2023) en una tabla LARGA (tidy) con un discriminador `desglose`.

| Fichero | desglose | dimensiones | territori |
|---------|----------|-------------|-----------|
| nacionalidad | `nacionalitat` | categoría (nacionalidad/región) | Cataluña |
| sexo+edad+tipo | `edat_delicte` | tipus_delicte × grup_edat × **sexe** | Cataluña |
| sexo+provincias | `regim_sexe` | régimen × sexe | Barcelona/Girona/Lleida/Tarragona/Cataluña |
| altas/bajas | `altes` / `baixes` | régimen × motiu × sexe | Cataluña |

**Output:** `data/clean/contexto_penitenciaria.csv`

**Esquema:** `anyo, id_territori, territori, desglose, categoria, subcategoria, sexe, grup_edat, valor`

**Es una tabla de CONTEXTO** (no fact del star): se une al análisis por `anyo + territori`.

⚠️ **REGLA DE ANÁLISIS:** cada fichero trae subtotales y totales inline (jerarquías).
NO sumar entre `desglose` distintos (cuentan los mismos reclusos de otra forma), ni
mezclar niveles dentro de un desglose (p.ej. 'Total' nacionalitat = española+extranjera;
o sexe 'Total' = 'Homes'+'Dones').

⚠️ **GOTCHAS de los ficheros raw** (resueltos en este notebook):
- **file 2 (edat_delicte)** tiene un eje de SEXO IMPLÍCITO: 3 bloques [Total/Homes/Dones]
  por año sin columna de sexo → se deriva por posición de bloque.
- **file 3 (regim_sexe)** son bloques fijos de 10 filas/año (parseo posicional).
- **file 4 (altes/baixes)** cambia de estructura: 2010-2011 usan fila 'Total' al final;
  2012-2023 ponen el total en la propia cabecera 'Altas'/'Bajas'.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
IDE = RAW / 'idescat'

# Mapa territori -> id_territorio canónico (dim_territorio fuente ministerio_ine)
dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
terr_canon = dim_territorio[dim_territorio['fuente'] == 'ministerio_ine']
MAP_TERR = {'Cataluña': int(terr_canon[terr_canon['nivel_territorial'] == 'ccaa']['id_territorio'].iloc[0])}
for prov in ['Barcelona', 'Girona', 'Lleida', 'Tarragona']:
    MAP_TERR[prov] = int(terr_canon[terr_canon['provincia'] == prov]['id_territorio'].iloc[0])
print('MAP_TERR:', MAP_TERR)

ESQUEMA = ['anyo', 'id_territori', 'territori', 'desglose', 'categoria', 'subcategoria', 'sexe', 'grup_edat', 'valor']

---
## 1. Nacionalidad → desglose `nacionalitat`

In [ ]:
f1 = pd.read_csv(IDE / 'idescat_poblacion_penitenciaria_nacionalidad_historico_2010_2023.csv', encoding='utf-8-sig')
f1 = f1.rename(columns={'Any': 'anyo', 'Nacionalitat': 'categoria', 'Valor': 'valor'})
f1['valor'] = pd.to_numeric(f1['valor'], errors='coerce')
f1['territori'] = 'Cataluña'
f1['id_territori'] = MAP_TERR['Cataluña']
f1['desglose'] = 'nacionalitat'
f1['subcategoria'] = pd.NA
f1['sexe'] = pd.NA
f1['grup_edat'] = pd.NA
pen_nac = f1[ESQUEMA].copy()

# Validación: Total == española + extranjera por año
for anyo, g in f1.groupby('anyo'):
    tot = g[g['categoria'] == 'Total']['valor'].sum()
    esp_ext = g[g['categoria'].isin(['Nacionalidad española', 'Nacionalidad extranjera'])]['valor'].sum()
    assert tot == esp_ext, f'{anyo}: Total {tot} != esp+ext {esp_ext}'
print(f'pen_nac: {len(pen_nac)} filas | validación Total=española+extranjera OK')

---
## 2. Sexo+edad+tipo → desglose `edat_delicte` (melt de columnas Edat)

In [ ]:
f2 = pd.read_csv(IDE / 'idescat_poblacion_penitenciaria_sexo_edad_tipo_historico_2010_2023.csv', encoding='utf-8-sig')
f2 = f2.rename(columns={'Any': 'anyo'})

# GOTCHA 1 — SEXO IMPLÍCITO: cada año son 3 bloques consecutivos del mismo tamaño con la
# misma lista de TipusDelicte, en orden [Total, Homes, Dones]. NO hay columna de sexo.
f2['pos'] = f2.groupby('anyo').cumcount()
year_len = f2.groupby('anyo')['anyo'].transform('size')
assert (year_len % 3 == 0).all(), 'Algún año no es divisible en 3 bloques de sexo'
block_size = year_len // 3
f2['sexe'] = (f2['pos'] // block_size).map({0: 'Total', 1: 'Homes', 2: 'Dones'})

# GOTCHA 2 — CÓDIGO PENAL padre: dentro de cada bloque hay 2 secciones, 'Delitos (antiguo
# Código penal)' y 'Delitos (nuevo Código penal)', cada una con su fila-total + sus delitos.
# Algunos delitos (p.ej. 'Contra la libertad sexual') existen en AMBOS códigos → se guarda
# el código penal padre (forward-fill) en 'subcategoria' para no dejar filas ambiguas.
PARENTS = {'Delitos (antiguo Código penal)', 'Delitos (nuevo Código penal)'}
f2['es_parent'] = f2['TipusDelicte'].isin(PARENTS)
f2['codi_penal'] = f2['TipusDelicte'].where(f2['es_parent'])
f2['codi_penal'] = f2.groupby(['anyo', 'sexe'])['codi_penal'].ffill()
f2['subcat'] = f2['codi_penal'].where(~f2['es_parent'])  # NA en las filas-total de cada código

MAP_EDAT = {
    'Edat_18_20': '18-20', 'Edat_21_25': '21-25', 'Edat_26_40': '26-40',
    'Edat_41_60': '41-60', 'Edat_61_mes': '61+', 'Total': 'Total'
}
f2_long = f2.melt(
    id_vars=['anyo', 'TipusDelicte', 'subcat', 'sexe'],
    value_vars=list(MAP_EDAT.keys()),
    var_name='grup_edat_raw', value_name='valor'
)
f2_long['grup_edat'] = f2_long['grup_edat_raw'].map(MAP_EDAT)
f2_long['valor'] = pd.to_numeric(f2_long['valor'], errors='coerce')
f2_long = f2_long.rename(columns={'TipusDelicte': 'categoria', 'subcat': 'subcategoria'})
f2_long['territori'] = 'Cataluña'
f2_long['id_territori'] = MAP_TERR['Cataluña']
f2_long['desglose'] = 'edat_delicte'
pen_edat = f2_long[ESQUEMA].copy()

# Validación 1: la columna 'Total' (suma de edades) de cada fila original cuadra
chk = f2.copy()
chk['suma_edades'] = chk[['Edat_18_20','Edat_21_25','Edat_26_40','Edat_41_60','Edat_61_mes']].sum(axis=1)
assert (chk['suma_edades'] == chk['Total']).all(), 'La columna Total no cuadra con la suma de edades'

# Validación 2 (SEXO): se afirma sobre grup_edat='Total' (todas las edades), que es
# perfectamente aditivo Homes+Dones=Total en todos los años → confirma la asignación de sexo.
# OJO: por edad individual hay micro-discrepancias (≤6 reclusos) por redondeo/confidencialidad
# de idescat; se reportan pero NO invalidan el desglose.
tot_edat = pen_edat[pen_edat['grup_edat'] == 'Total']
piv_sex = tot_edat.pivot_table(index='anyo', columns='sexe', values='valor', aggfunc='sum')
assert ((piv_sex['Homes'] + piv_sex['Dones']) == piv_sex['Total']).all(), 'sexo no cuadra en grup_edat=Total'
leaves = pen_edat[pen_edat['subcategoria'].notna()]
chk2 = leaves.pivot_table(index=['anyo', 'grup_edat'], columns='sexe', values='valor', aggfunc='sum')
max_dif = int((chk2['Homes'] + chk2['Dones'] - chk2['Total']).abs().max())

sexes_edat = sorted(pen_edat['sexe'].unique())
print(f'pen_edat: {len(pen_edat)} filas | sexes: {sexes_edat}')
print(f'  validación sexo (grup_edat=Total) OK | micro-discrepancia máx por edad (idescat): {max_dif} reclusos')

---
## 3. Régimen+sexo+provincias → desglose `regim_sexe` (bloques de 10 filas/año + melt provincias)

In [ ]:
f3 = pd.read_csv(IDE / 'idescat_poblacion_penitenciaria_sexo_historico_2010_2023.csv', encoding='utf-8-sig')

# Cada año es un bloque FIJO de 10 filas con este patrón posicional:
#   [Penados(Total,H,M), Preventius(Total,H,M), Altres règims(Total,H,M), Total general]
REGIM_POS = {0:'Penados',1:'Penados',2:'Penados', 3:'Preventius',4:'Preventius',5:'Preventius',
             6:'Altres règims',7:'Altres règims',8:'Altres règims', 9:'Total general'}
SEXE_POS  = {0:'Total',1:'Homes',2:'Dones', 3:'Total',4:'Homes',5:'Dones',
             6:'Total',7:'Homes',8:'Dones', 9:'Total'}

f3 = f3.rename(columns={'Any': 'anyo'})
f3['pos'] = f3.groupby('anyo').cumcount()
# Verificar que todos los años tienen exactamente 10 filas
assert (f3.groupby('anyo').size() == 10).all(), 'Algún año no tiene 10 filas en file 3'
f3['categoria'] = f3['pos'].map(REGIM_POS)
f3['sexe'] = f3['pos'].map(SEXE_POS)

prov_cols = ['Barcelona', 'Girona', 'Lleida', 'Tarragona', 'Cataluña']
f3_long = f3.melt(
    id_vars=['anyo', 'categoria', 'sexe'],
    value_vars=prov_cols,
    var_name='territori', value_name='valor'
)
f3_long['valor'] = pd.to_numeric(f3_long['valor'], errors='coerce')
f3_long['id_territori'] = f3_long['territori'].map(MAP_TERR)
f3_long['desglose'] = 'regim_sexe'
f3_long['subcategoria'] = pd.NA
f3_long['grup_edat'] = pd.NA
pen_regim = f3_long[ESQUEMA].copy()

# Validación 1: en los 3 régimenes con desglose por sexo → Homes + Dones == Total del régimen
REGIM_3 = ['Penados', 'Preventius', 'Altres règims']
piv = (f3_long[f3_long['categoria'].isin(REGIM_3)]
       .pivot_table(index=['anyo','territori','categoria'], columns='sexe', values='valor', aggfunc='sum'))
desc = piv[(piv['Homes'] + piv['Dones']) != piv['Total']]
assert len(desc) == 0, 'Homes+Dones no cuadra con Total del régimen'

# Validación 2: 'Total general' (sexe=Total) == suma de los 3 régimenes (sexe=Total) por territori
tot_sexe = f3_long[f3_long['sexe'] == 'Total']
for (anyo, terr), g in tot_sexe.groupby(['anyo', 'territori']):
    gt = g[g['categoria'] == 'Total general']['valor'].sum()
    suma = g[g['categoria'].isin(REGIM_3)]['valor'].sum()
    assert gt == suma, f'{anyo}/{terr}: Total general {gt} != suma régimenes {suma}'

print(f'pen_regim: {len(pen_regim)} filas | validaciones Homes+Dones=Total y Total general=suma régimenes OK')

---
## 4. Altas/Bajas → desglose `altes` / `baixes` (forward-fill de sección y régimen)

In [ ]:
f4 = pd.read_csv(IDE / 'idescat_altas_bajas_penitenciarias_historico_2010_2023.csv', encoding='utf-8-sig')
f4 = f4.rename(columns={'Any': 'anyo'})

FLUX_HDR = {'Altas': 'altes', 'Bajas': 'baixes'}
REGIM = {'Penados', 'Preventivos', 'Internados judiciales'}

# OJO: la estructura cambia entre años:
#  - 2010-2011: 'Altas'/'Bajas' son cabeceras VACÍAS + una fila 'Total' al final de la sección.
#  - 2012-2023: la fila 'Altas'/'Bajas' YA contiene el total de la sección (no hay fila 'Total').
# Se maneja: el total de sección ('Total general') se emite desde donde esté.

def emit(registros, row, flux, categoria, subcat):
    for sexe, col in [('Homes', 'Homes'), ('Dones', 'Dones'), ('Total', 'Total')]:
        registros.append({
            'anyo': row['anyo'], 'id_territori': MAP_TERR['Cataluña'], 'territori': 'Cataluña',
            'desglose': flux, 'categoria': categoria, 'subcategoria': subcat,
            'sexe': sexe, 'grup_edat': pd.NA, 'valor': pd.to_numeric(row[col], errors='coerce')
        })

registros = []
flux, regim = None, None
for _, r in f4.iterrows():
    cat = r['Categoria']
    if cat in FLUX_HDR:
        flux = FLUX_HDR[cat]; regim = None
        if pd.notna(r['Total']):          # 2012+: la cabecera lleva el total de la sección
            emit(registros, r, flux, 'Total general', pd.NA)
        continue
    if cat == 'Total':                    # 2010-2011: total al final de la sección
        emit(registros, r, flux, 'Total general', pd.NA)
        continue
    if cat in REGIM:                      # total del régimen
        regim = cat
        emit(registros, r, flux, cat, pd.NA)
    else:                                 # motivo dentro del régimen actual
        emit(registros, r, flux, regim, cat)

pen_flux = pd.DataFrame(registros)[ESQUEMA]
pen_flux = pen_flux.dropna(subset=['valor'])

# Validación: por (año, flux) → 'Total general' (sexe=Total) == suma régimenes (sexe=Total)
for (anyo, flux_v), g in pen_flux[pen_flux['sexe'] == 'Total'].groupby(['anyo', 'desglose']):
    tot = g[g['categoria'] == 'Total general']['valor'].sum()
    reg = g[(g['categoria'].isin(REGIM)) & (g['subcategoria'].isna())]['valor'].sum()
    assert tot == reg, f'{anyo}/{flux_v}: Total general {tot} != suma régimenes {reg}'
print(f'pen_flux: {len(pen_flux)} filas | validación Total general = suma régimenes OK')
print('Desgloses flux:', sorted(pen_flux['desglose'].unique().tolist()))

---
## 5. Unir y guardar

In [ ]:
contexto = pd.concat([pen_nac, pen_edat, pen_regim, pen_flux], ignore_index=True)
contexto['anyo'] = contexto['anyo'].astype(int)
contexto['id_territori'] = contexto['id_territori'].astype(int)
contexto['valor'] = contexto['valor'].astype('Int64')  # nullable por si algún melt deja NA

print('contexto_penitenciaria:', contexto.shape)
print('\nFilas por desglose:')
print(contexto['desglose'].value_counts())
print('\nRango años:', contexto['anyo'].min(), '-', contexto['anyo'].max())
print('Territoris:', sorted(contexto['territori'].unique()))
print('id_territori en dim_territorio:', contexto['id_territori'].isin(dim_territorio['id_territorio']).all())
print('Nulos en valor:', contexto['valor'].isna().sum())
contexto.head()

In [ ]:
ruta_salida = CLEAN / 'contexto_penitenciaria.csv'
contexto.to_csv(ruta_salida, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta_salida.name}: {len(contexto)} filas')
print('\nListo para continuar con 06_etl_socioeconomico.ipynb')